In [3]:
import pandas as pd

In [4]:
results=pd.read_csv('results.csv')

In [5]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
shootouts=pd.read_csv('shootouts.csv')

In [7]:
shootouts.head()

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


In [8]:
results['date']=pd.to_datetime(results['date'])
modern_results=results[results['date'].dt.year>=1998].copy()

In [9]:
shootouts['date']=pd.to_datetime(shootouts['date'])
modern_shootouts=shootouts[shootouts['date'].dt.year>=1998].copy().drop(columns=['first_shooter'])

In [10]:
modern_results.shape[0]

26853

In [11]:
modern_shootouts.shape[0]

444

In [12]:
modern_results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
22525,1998-01-03,Malawi,Zimbabwe,1.0,0.0,Friendly,Blantyre,Malawi,False
22526,1998-01-04,Barbados,Trinidad and Tobago,0.0,1.0,Friendly,Bridgetown,Barbados,False
22527,1998-01-04,Burkina Faso,Mozambique,1.0,0.0,Friendly,Bobo Dioulasso,Burkina Faso,False
22528,1998-01-04,Guinea,Togo,1.0,0.0,Friendly,Conakry,Guinea,False
22529,1998-01-04,Malawi,Zimbabwe,0.0,2.0,Friendly,Lilongwe,Malawi,False


In [13]:
modern_shootouts.head()

,date,home_team,away_team,winner
231,1998-01-31,Egypt,South Korea,South Korea
232,1998-01-31,Iran,Chile,Iran
233,1998-02-21,Burkina Faso,Tunisia,Burkina Faso
234,1998-02-21,Ivory Coast,Egypt,Egypt
235,1998-02-27,Burkina Faso,DR Congo,DR Congo


In [14]:
def determine_interim_winner(row):
  if row['home_score']>row['away_score']:
    return row['home_team']
  elif row['home_score']<row['away_score']:
    return row['away_team']
  else:
    return 'Draw'

modern_results['final_outcome']=modern_results.apply(determine_interim_winner,axis=1)

merged_data=pd.merge(modern_results,modern_shootouts,how='left',
                     on=['date','home_team','away_team'], suffixes=('', '_shootouts'))
merged_data.loc[merged_data['winner'].notna(), 'final_outcome']=merged_data['winner']

In [15]:
final_dataset=merged_data.drop(columns=['winner'])  #dropped the winner column
final_dataset.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,final_outcome
0,1998-01-03,Malawi,Zimbabwe,1.0,0.0,Friendly,Blantyre,Malawi,False,Malawi
1,1998-01-04,Barbados,Trinidad and Tobago,0.0,1.0,Friendly,Bridgetown,Barbados,False,Trinidad and Tobago
2,1998-01-04,Burkina Faso,Mozambique,1.0,0.0,Friendly,Bobo Dioulasso,Burkina Faso,False,Burkina Faso
3,1998-01-04,Guinea,Togo,1.0,0.0,Friendly,Conakry,Guinea,False,Guinea
4,1998-01-04,Malawi,Zimbabwe,0.0,2.0,Friendly,Lilongwe,Malawi,False,Zimbabwe


In [16]:
final_dataset[final_dataset['date']=='2022-12-18']

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,final_outcome
23260,2022-12-18,Argentina,France,3.0,3.0,FIFA World Cup,Lusail,Qatar,True,Argentina


In [17]:
final_dataset['final_outcome'].isna().sum()

np.int64(0)

In [18]:
import numpy as np

# 1. Ensure the data is sorted chronologically
final_modern_dataset = final_dataset.sort_values('date').reset_index(drop=True)

In [19]:
import numpy as np
import pandas as pd

# 1. Critical Step: Sort chronologically so index sequence matches time sequence
final_modern_dataset = final_modern_dataset.sort_values(by='date').reset_index(drop=True)

# 2. Re-initialize clean tracking lists
home_form_scored, home_class_scored = [], []
home_form_conceded, home_class_conceded = [], []
away_form_scored, away_class_scored = [], []
away_form_conceded, away_class_conceded = [], []

# 3. Step through the dataset using a strict historical sequence index
for current_idx, row in final_modern_dataset.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']

    # Capture ALL historical rows that happened BEFORE this row index
    past_data = final_modern_dataset.iloc[:current_idx]

    # --- Home Team History ---
    home_history = past_data[(past_data['home_team'] == home_team) | (past_data['away_team'] == home_team)]
    if len(home_history) >= 1:
        home_scored = np.where(home_history['home_team'] == home_team, home_history['home_score'], home_history['away_score'])
        home_conceded = np.where(home_history['home_team'] == home_team, home_history['away_score'], home_history['home_score'])

        home_form_scored.append(home_scored[-5:].mean())
        home_form_conceded.append(home_conceded[-5:].mean())
        home_class_scored.append(home_scored[-20:].mean() if len(home_scored) >= 20 else home_scored.mean())
        home_class_conceded.append(home_conceded[-20:].mean() if len(home_conceded) >= 20 else home_conceded.mean())
    else:
        # True historical fallback values if a team is completely new to the modern system
        home_form_scored.append(0); home_form_conceded.append(0)
        home_class_scored.append(0); home_class_conceded.append(0)

    # --- Away Team History ---
    away_history = past_data[(past_data['home_team'] == away_team) | (past_data['away_team'] == away_team)]
    if len(away_history) >= 1:
        away_scored = np.where(away_history['home_team'] == away_team, away_history['home_score'], away_history['away_score'])
        away_conceded = np.where(away_history['home_team'] == away_team, away_history['away_score'], away_history['home_score'])

        away_form_scored.append(away_scored[-5:].mean())
        away_form_conceded.append(away_conceded[-5:].mean())
        away_class_scored.append(away_scored[-20:].mean() if len(away_scored) >= 20 else away_scored.mean())
        away_class_conceded.append(away_conceded[-20:].mean() if len(away_conceded) >= 20 else away_conceded.mean())
    else:
        away_form_scored.append(0); away_form_conceded.append(0)
        away_class_scored.append(0); away_class_conceded.append(0)

# 4. Attach the calculated sequences back to your main DataFrame
final_modern_dataset['home_form_scored'] = home_form_scored
final_modern_dataset['home_form_conceded'] = home_form_conceded
final_modern_dataset['home_class_scored'] = home_class_scored
final_modern_dataset['home_class_conceded'] = home_class_conceded

final_modern_dataset['away_form_scored'] = away_form_scored
final_modern_dataset['away_form_conceded'] = away_form_conceded
final_modern_dataset['away_class_scored'] = away_class_scored
final_modern_dataset['away_class_conceded'] = away_class_conceded

In [20]:
# 1. Masks first
world_cup_kickoff = '2026-06-11'
final_modern_dataset['date'] = pd.to_datetime(final_modern_dataset['date'])
train_mask = final_modern_dataset['date'] < world_cup_kickoff
test_mask  = final_modern_dataset['date'] >= world_cup_kickoff

# 2. Null out future outcomes BEFORE encode_target runs
final_modern_dataset.loc[test_mask, 'final_outcome'] = None

# 3. Encode target (WC rows will now be NaN, not 1)
def encode_target(row):
    if pd.isna(row['final_outcome']):
        return None
    elif row['final_outcome'] == row['home_team']:
        return 2
    elif row['final_outcome'] == 'Draw':
        return 1
    else:
        return 0

final_modern_dataset['target'] = final_modern_dataset.apply(encode_target, axis=1)

# 4. Frozen stats fill for WC matches
pre_wc_data = final_modern_dataset[train_mask]
team_last_stats = {}
for team in set(pre_wc_data['home_team']).union(set(pre_wc_data['away_team'])):
    team_matches = pre_wc_data[
        (pre_wc_data['home_team'] == team) | (pre_wc_data['away_team'] == team)
    ]
    if len(team_matches) == 0:
        continue
    last = team_matches.iloc[-1]
    side = 'home' if last['home_team'] == team else 'away'
    team_last_stats[team] = {
        'form_scored':    last[f'{side}_form_scored'],
        'form_conceded':  last[f'{side}_form_conceded'],
        'class_scored':   last[f'{side}_class_scored'],
        'class_conceded': last[f'{side}_class_conceded'],
    }

for idx, row in final_modern_dataset[test_mask].iterrows():
    for side, team in [('home', row['home_team']), ('away', row['away_team'])]:
        if pd.isna(row[f'{side}_form_scored']) and team in team_last_stats:
            stats = team_last_stats[team]
            final_modern_dataset.at[idx, f'{side}_form_scored']   = stats['form_scored']
            final_modern_dataset.at[idx, f'{side}_form_conceded'] = stats['form_conceded']
            final_modern_dataset.at[idx, f'{side}_class_scored']  = stats['class_scored']
            final_modern_dataset.at[idx, f'{side}_class_conceded']= stats['class_conceded']

# 5. Fallback for any remaining NaNs (brand new teams with zero history)
feature_columns = [
    'home_form_scored', 'home_form_conceded', 'home_class_scored', 'home_class_conceded',
    'away_form_scored', 'away_form_conceded', 'away_class_scored', 'away_class_conceded'
]
train_means = final_modern_dataset.loc[train_mask, feature_columns].mean()
final_modern_dataset[feature_columns] = final_modern_dataset[feature_columns].fillna(train_means)

# 6. Rebuild X and y
X = final_modern_dataset[feature_columns]
y = final_modern_dataset['target']

In [21]:
final_modern_dataset[final_modern_dataset['date'] == '2026-06-11']

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,final_outcome,home_form_scored,home_form_conceded,home_class_scored,home_class_conceded,away_form_scored,away_form_conceded,away_class_scored,away_class_conceded,target
26781,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,None,1.6,0.2,1.25,0.65,1.2,1.4,1.65,0.75,NaN
26782,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,None,1.4,1.0,1.75,0.95,2.6,1.0,1.80,1.20,NaN


In [22]:
X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print(f" Total Historical Training Matches: {X_train.shape[0]}")
print(f" Live World Cup Tournament Matches to Predict: {X_test.shape[0]}")

 Total Historical Training Matches: 26781
 Live World Cup Tournament Matches to Predict: 72


In [23]:
print(" Missing values remaining in the updated X_test:", X_test.isna().sum().sum())
print(f"Shapes look good: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

 Missing values remaining in the updated X_test: 0
Shapes look good: Train=26781, Test=72


In [24]:
final_modern_dataset[(final_modern_dataset['home_team']=='Portugal') |
                     (final_modern_dataset['away_team']=='Portugal')]
# Portugal will win the World Cup

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,final_outcome,home_form_scored,home_form_conceded,home_class_scored,home_class_conceded,away_form_scored,away_form_conceded,away_class_scored,away_class_conceded,target
198,1998-04-22,England,Portugal,3.0,0.0,Friendly,London,England,False,England,0.500000,1.500000,0.500000,1.500000,0.00,0.0,0.000000,0.000000,2.0
463,1998-08-19,Portugal,Mozambique,2.0,1.0,Friendly,Ponta Delgada,Portugal,False,Portugal,0.000000,3.000000,0.000000,3.000000,1.00,0.8,0.833333,1.722222,2.0
519,1998-09-06,Hungary,Portugal,1.0,3.0,UEFA Euro qualification,Budapest,Hungary,False,Portugal,1.600000,0.600000,1.600000,0.600000,1.00,2.0,1.000000,2.000000,0.0
606,1998-10-10,Portugal,Romania,0.0,1.0,UEFA Euro qualification,Porto,Portugal,False,Romania,1.666667,1.666667,1.666667,1.666667,1.80,0.6,1.916667,0.833333,0.0
625,1998-10-14,Slovakia,Portugal,0.0,3.0,UEFA Euro qualification,Bratislava,Slovakia,False,Portugal,1.400000,0.200000,1.333333,0.666667,1.25,1.5,1.250000,1.500000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26651,2026-03-28,Mexico,Portugal,0.0,0.0,Friendly,Mexico City,Mexico,False,Draw,1.400000,0.400000,1.550000,0.850000,3.00,1.4,2.250000,1.000000,1.0
26720,2026-03-31,United States,Portugal,0.0,2.0,Friendly,Atlanta,United States,False,Portugal,2.400000,1.800000,1.950000,1.450000,2.40,1.0,2.100000,1.000000,0.0
26801,2026-06-17,Portugal,DR Congo,NaN,NaN,FIFA World Cup,Houston,United States,True,None,2.600000,1.000000,2.200000,0.900000,1.20,0.2,1.400000,0.550000,NaN
26826,2026-06-23,Portugal,Uzbekistan,NaN,NaN,FIFA World Cup,Houston,United States,True,None,2.400000,1.000000,2.100000,1.000000,1.40,0.6,1.350000,0.650000,NaN


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

import joblib
joblib.dump(scaler, 'feature_scaler.pkl')

rf_model=RandomForestClassifier(n_estimators=200, max_depth=15,
                                min_samples_split=5, random_state=42)
rf_model.fit(X_train, y_train)

train_preds=rf_model.predict(X_train)
accuracy_score(y_train, train_preds)

0.7891788954856055

In [26]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(n_estimators=150, max_depth=10,
                          learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_model.score(X_train, y_train)

0.7248795788058698

In [27]:
print(classification_report(y_train, train_preds))

              precision    recall  f1-score   support

         0.0       0.85      0.74      0.79      7828
         1.0       1.00      0.45      0.62      5827
         2.0       0.73      0.97      0.83     13126

    accuracy                           0.79     26781
   macro avg       0.86      0.72      0.75     26781
weighted avg       0.82      0.79      0.78     26781



Random Forest is Overfitting the training set

In [28]:
rf_test_preds=rf_model.predict(X_test)
xgb_test_preds=xgb_model.predict(X_test)

matches_agreed=(rf_test_preds == xgb_test_preds).sum()
print(f'Out of 72 matches, the models agree on: {matches_agreed} games')

Out of 72 matches, the models agree on: 67 games


In [29]:
simulation_results = final_modern_dataset[test_mask].copy()
print("\n--- First 5 World Cup Match Probabilities (XGBoost) ---")
for i in range(5):
    home = simulation_results.iloc[i]['home_team']
    away = simulation_results.iloc[i]['away_team']
    probs = xgb_model.predict_proba(X_test)[i]
    print(f"{home} vs {away} -> Home Win: {probs[2]:.1%}, Draw: {probs[1]:.1%}, Away Win: {probs[0]:.1%}")


--- First 5 World Cup Match Probabilities (XGBoost) ---
Mexico vs South Africa -> Home Win: 54.2%, Draw: 26.6%, Away Win: 19.3%
South Korea vs Czech Republic -> Home Win: 52.9%, Draw: 22.6%, Away Win: 24.4%
Canada vs Bosnia and Herzegovina -> Home Win: 66.6%, Draw: 20.2%, Away Win: 13.2%
United States vs Paraguay -> Home Win: 36.6%, Draw: 34.0%, Away Win: 29.4%
Brazil vs Morocco -> Home Win: 31.5%, Draw: 33.1%, Away Win: 35.3%


In [30]:
final_modern_dataset.to_csv('world_cup_processed.csv', index=False)

import joblib
joblib.dump(xgb_model, 'baseline_xgb_model.pkl')

['baseline_xgb_model.pkl']

In [31]:
joblib.dump(X_train, 'X_train.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(y_test, 'y_test.pkl')

['y_test.pkl']